# Dataset Loading and Inspection

In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

try:
    import librosa
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False

DATASET_PATH = (r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Dataset")

IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
)

AUDIO_EXTENSIONS = (
    ".wav", ".mp3", ".flac",
    ".ogg", ".m4a", ".aac"
)

CSV_EXTENSIONS = (
    ".csv",
)

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"\nDataset path does not exist:\n{DATASET_PATH}"
    )

all_files = []

for root, dirs, files in os.walk(DATASET_PATH):

    for file in files:

        file_path = os.path.join(root, file)

        all_files.append(file_path)

image_files = []
audio_files = []
csv_files = []
other_files = []

for file_path in all_files:

    extension = os.path.splitext(file_path)[1].lower()

    if extension in IMAGE_EXTENSIONS:

        image_files.append(file_path)

    elif extension in AUDIO_EXTENSIONS:

        audio_files.append(file_path)

    elif extension in CSV_EXTENSIONS:

        csv_files.append(file_path)

    else:

        other_files.append(file_path)

df = None
csv_file_used = None

if len(csv_files) > 0:

    # Prefer the first CSV file
    csv_file_used = csv_files[0]

    try:

        df = pd.read_csv(csv_file_used)

    except UnicodeDecodeError:

        df = pd.read_csv(
            csv_file_used,
            encoding="latin1"
        )

    except Exception as e:

        print("\nWARNING: Could not read CSV file.")
        print("File :", csv_file_used)
        print("Error:", e)

        df = None

print("\n" + "=" * 80)
print("FINAL DATASET SUMMARY")
print("=" * 80)

print(
    f"""
Total Files        : {len(all_files)}
Image Files        : {len(image_files)}
Audio Files        : {len(audio_files)}
CSV Files          : {len(csv_files)}
Other Files        : {len(other_files)}

Pillow Available   : {PIL_AVAILABLE}
Librosa Available  : {LIBROSA_AVAILABLE}
"""
)


# ============================================================
# 9. CSV INFORMATION
# ============================================================

if df is not None:

    print("=" * 80)
    print("CSV DATASET INFORMATION")
    print("=" * 80)

    print(
        "CSV Shape         :",
        df.shape
    )

    print(
        "CSV Columns       :",
        len(df.columns)
    )

    print(
        "CSV Rows          :",
        len(df)
    )

    TARGET_COLUMN = "Dissemination_Effectiveness"

    if TARGET_COLUMN in df.columns:

        print("\n" + "-" * 80)
        print("TARGET INFORMATION")
        print("-" * 80)

        print(
            "\nTarget            :",
            TARGET_COLUMN
        )

        print(
            "Number of Classes :",
            df[TARGET_COLUMN].nunique()
        )

        print("\nClass Distribution:")

        print(
            df[TARGET_COLUMN]
            .value_counts(dropna=False)
        )

        print("\nClass Percentages:")

        percentages = (
            df[TARGET_COLUMN]
            .value_counts(normalize=True, dropna=False)
            .mul(100)
            .round(2)
        )

        print(percentages)


    else:

        print(
            f"\nTarget column "
            f"'{TARGET_COLUMN}' was not found."
        )

    print("\n" + "-" * 80)
    print("MISSING VALUE INFORMATION")
    print("-" * 80)

    missing_values = df.isnull().sum()

    total_missing = missing_values.sum()

    print(
        "\nTotal Missing Values:",
        total_missing
    )

    if total_missing > 0:

        print("\nColumns With Missing Values:")

        missing_table = (
            missing_values[
                missing_values > 0
            ]
            .sort_values(ascending=False)
        )

        print(missing_table)

    else:

        print(
            "\nNo missing values found."
        )

    print("\n" + "-" * 80)
    print("DUPLICATE INFORMATION")
    print("-" * 80)

    duplicate_count = df.duplicated().sum()

    print(
        "\nDuplicate Rows:",
        duplicate_count
    )

print("\n" + "=" * 80)
print("DATASET INSPECTION COMPLETED")
print("=" * 80)


FINAL DATASET SUMMARY

Total Files        : 59
Image Files        : 50
Audio Files        : 8
CSV Files          : 1
Other Files        : 0

Pillow Available   : True
Librosa Available  : True

CSV DATASET INFORMATION
CSV Shape         : (7000, 38)
CSV Columns       : 38
CSV Rows          : 7000

--------------------------------------------------------------------------------
TARGET INFORMATION
--------------------------------------------------------------------------------

Target            : Dissemination_Effectiveness
Number of Classes : 5

Class Distribution:
Dissemination_Effectiveness
Very High    1400
Moderate     1400
Low          1400
Very Low     1400
High         1400
Name: count, dtype: int64

Class Percentages:
Dissemination_Effectiveness
Very High    20.0
Moderate     20.0
Low          20.0
Very Low     20.0
High         20.0
Name: proportion, dtype: float64

--------------------------------------------------------------------------------
MISSING VALUE INFORMATION
-----

# Preprocessing 

In [2]:
!pip install pandas numpy pillow matplotlib librosa soundfile scikit-learn torch torchvision sentence-transformers tqdm


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
import subprocess
import importlib


def install_if_missing(package_name, import_name=None):

    if import_name is None:
        import_name = package_name

    try:
        importlib.import_module(import_name)

    except ImportError:

        print(
            f"Installing missing package: {package_name}"
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package_name
            ]
        )


# Required packages
install_if_missing("numpy")
install_if_missing("pandas")
install_if_missing("Pillow", "PIL")
install_if_missing("tqdm")
install_if_missing("matplotlib")
install_if_missing("librosa")
install_if_missing("scikit-learn", "sklearn")
install_if_missing(
    "sentence-transformers",
    "sentence_transformers"
)

import os
import re
import json
import random
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

from PIL import (
    Image,
    ImageEnhance
)

import librosa
import librosa.display

import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sentence_transformers import (
    SentenceTransformer
)


warnings.filterwarnings(
    "ignore"
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATASET_PATH = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Dataset"
)

OUTPUT_PATH = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output"
)

IMAGE_OUTPUT = (
    OUTPUT_PATH /
    "Images"
)

AUDIO_OUTPUT = (
    OUTPUT_PATH /
    "Audios"
)

SPECTROGRAM_OUTPUT = (
    AUDIO_OUTPUT /
    "Spectrograms"
)

CSV_OUTPUT = (
    OUTPUT_PATH /
    "CSV"
)

EMBEDDING_OUTPUT = (
    CSV_OUTPUT /
    "Embeddings"
)


for folder in [
    OUTPUT_PATH,
    IMAGE_OUTPUT,
    AUDIO_OUTPUT,
    SPECTROGRAM_OUTPUT,
    CSV_OUTPUT,
    EMBEDDING_OUTPUT
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("\n")
print("=" * 75)
print("MULTIMODAL DATA PREPROCESSING")
print("=" * 75)

print("\nDataset path:")
print(DATASET_PATH)

print("\nOutput path:")
print(OUTPUT_PATH)

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset directory does not exist:\n{DATASET_PATH}"
    )

print("\n")
print("=" * 75)
print("DATASET FILE SUMMARY")
print("=" * 75)

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}


image_files = []

for root, dirs, files in os.walk(
    DATASET_PATH
):

    for filename in files:

        path = Path(root) / filename

        if (
            path.suffix.lower()
            in image_extensions
        ):

            if (
                path.exists()
                and path.is_file()
            ):

                image_files.append(
                    path
                )


image_files = sorted(
    list(
        dict.fromkeys(
            image_files
        )
    ),
    key=lambda x: str(x).lower()
)

audio_extensions = {
    ".mp3",
    ".wav",
    ".m4a",
    ".aac",
    ".flac"
}


audio_files = []

for root, dirs, files in os.walk(
    DATASET_PATH
):

    for filename in files:

        path = Path(root) / filename

        if (
            path.suffix.lower()
            in audio_extensions
        ):

            if (
                path.exists()
                and path.is_file()
            ):

                audio_files.append(
                    path
                )


audio_files = sorted(
    list(
        dict.fromkeys(
            audio_files
        )
    ),
    key=lambda x: str(x).lower()
)

csv_files = []

for root, dirs, files in os.walk(
    DATASET_PATH
):

    for filename in files:

        path = Path(root) / filename

        if (
            path.suffix.lower()
            == ".csv"
        ):

            if (
                path.exists()
                and path.is_file()
            ):

                csv_files.append(
                    path
                )


csv_files = sorted(
    list(
        dict.fromkeys(
            csv_files
        )
    ),
    key=lambda x: str(x).lower()
)


print(
    "\nImages found :",
    len(image_files)
)

print(
    "Audios found :",
    len(audio_files)
)

print(
    "CSV files    :",
    len(csv_files)
)


if len(image_files) == 0:

    raise RuntimeError(
        "No image files were found."
    )


if len(audio_files) == 0:

    raise RuntimeError(
        "No audio files were found."
    )


if len(csv_files) == 0:

    raise RuntimeError(
        "No CSV files were found."
    )

print("\n")
print("=" * 75)
print("IMAGE PREPROCESSING")
print("=" * 75)

IMAGE_SIZE = (
    224,
    224
)

IMAGE_OUTPUTS_PER_INPUT = 4

image_metadata = []

def resize_image_keep_ratio(
    image,
    target_size
):

    target_width, target_height = (
        target_size
    )

    width, height = image.size

    scale = max(
        target_width / width,
        target_height / height
    )

    new_width = int(
        width * scale
    )

    new_height = int(
        height * scale
    )

    image = image.resize(
        (
            new_width,
            new_height
        ),
        Image.Resampling.LANCZOS
    )

    return image

def center_crop(
    image,
    target_size
):

    target_width, target_height = (
        target_size
    )

    width, height = image.size

    left = max(
        0,
        (width - target_width) // 2
    )

    top = max(
        0,
        (height - target_height) // 2
    )

    right = left + target_width

    bottom = top + target_height

    return image.crop(
        (
            left,
            top,
            right,
            bottom
        )
    )

def random_crop(
    image,
    target_size
):

    target_width, target_height = (
        target_size
    )

    width, height = image.size

    if (
        width < target_width
        or height < target_height
    ):

        image = resize_image_keep_ratio(
            image,
            target_size
        )

        width, height = image.size


    max_left = (
        width - target_width
    )

    max_top = (
        height - target_height
    )


    if max_left > 0:

        left = random.randint(
            0,
            max_left
        )

    else:

        left = 0


    if max_top > 0:

        top = random.randint(
            0,
            max_top
        )

    else:

        top = 0


    return image.crop(
        (
            left,
            top,
            left + target_width,
            top + target_height
        )
    )

def intensity_normalization(
    image
):

    image_array = np.asarray(
        image
    ).astype(
        np.float32
    )

    image_array /= 255.0

    image_array = np.clip(
        image_array,
        0.0,
        1.0
    )

    image_array = (
        image_array * 255.0
    ).astype(
        np.uint8
    )

    return Image.fromarray(
        image_array
    )

def apply_color_jitter(
    image
):

    # Brightness
    brightness_factor = random.uniform(
        0.85,
        1.15
    )

    image = ImageEnhance.Brightness(
        image
    ).enhance(
        brightness_factor
    )


    # Contrast
    contrast_factor = random.uniform(
        0.85,
        1.15
    )

    image = ImageEnhance.Contrast(
        image
    ).enhance(
        contrast_factor
    )


    # Color
    color_factor = random.uniform(
        0.85,
        1.15
    )

    image = ImageEnhance.Color(
        image
    ).enhance(
        color_factor
    )


    return image

total_image_outputs = 0


for image_index, image_path in enumerate(

    tqdm(
        image_files,
        desc="Processing images"
    ),

    start=1

):

    try:

        image = Image.open(
            image_path
        ).convert(
            "RGB"
        )


        original_width, original_height = (
            image.size
        )

        resized_image = (
            resize_image_keep_ratio(
                image,
                IMAGE_SIZE
            )
        )

        processed_image = center_crop(
            resized_image,
            IMAGE_SIZE
        )

        processed_image = (
            intensity_normalization(
                processed_image
            )
        )


        output_name = (
            f"image_{image_index:03d}_"
            f"original.png"
        )

        output_path = (
            IMAGE_OUTPUT /
            output_name
        )

        processed_image.save(
            output_path,
            quality=95
        )


        image_metadata.append({

            "Image_ID":
                f"image_{image_index:03d}_original",

            "Original_File":
                str(image_path),

            "Output_File":
                str(output_path),

            "Original_Width":
                original_width,

            "Original_Height":
                original_height,

            "Output_Width":
                IMAGE_SIZE[0],

            "Output_Height":
                IMAGE_SIZE[1],

            "Transformation":
                "Resize + Center Crop + Intensity Normalization"

        })


        total_image_outputs += 1

        aug_image = random_crop(
            resized_image,
            IMAGE_SIZE
        )

        aug_image = (
            intensity_normalization(
                aug_image
            )
        )

        output_name = (
            f"image_{image_index:03d}_"
            f"crop.png"
        )

        output_path = (
            IMAGE_OUTPUT /
            output_name
        )

        aug_image.save(
            output_path,
            quality=95
        )


        image_metadata.append({

            "Image_ID":
                f"image_{image_index:03d}_crop",

            "Original_File":
                str(image_path),

            "Output_File":
                str(output_path),

            "Original_Width":
                original_width,

            "Original_Height":
                original_height,

            "Output_Width":
                IMAGE_SIZE[0],

            "Output_Height":
                IMAGE_SIZE[1],

            "Transformation":
                "Resize + Random Crop + Intensity Normalization"

        })


        total_image_outputs += 1


        flip_image = (
            resized_image
            .transpose(
                Image.Transpose.FLIP_LEFT_RIGHT
            )
        )

        flip_image = center_crop(
            flip_image,
            IMAGE_SIZE
        )

        flip_image = (
            intensity_normalization(
                flip_image
            )
        )

        output_name = (
            f"image_{image_index:03d}_"
            f"flip.png"
        )

        output_path = (
            IMAGE_OUTPUT /
            output_name
        )

        flip_image.save(
            output_path,
            quality=95
        )


        image_metadata.append({

            "Image_ID":
                f"image_{image_index:03d}_flip",

            "Original_File":
                str(image_path),

            "Output_File":
                str(output_path),

            "Original_Width":
                original_width,

            "Original_Height":
                original_height,

            "Output_Width":
                IMAGE_SIZE[0],

            "Output_Height":
                IMAGE_SIZE[1],

            "Transformation":
                "Resize + Horizontal Flip + Intensity Normalization"

        })


        total_image_outputs += 1

        jitter_image = random_crop(
            resized_image,
            IMAGE_SIZE
        )

        jitter_image = apply_color_jitter(
            jitter_image
        )

        jitter_image = (
            intensity_normalization(
                jitter_image
            )
        )

        output_name = (
            f"image_{image_index:03d}_"
            f"jitter.png"
        )

        output_path = (
            IMAGE_OUTPUT /
            output_name
        )

        jitter_image.save(
            output_path,
            quality=95
        )


        image_metadata.append({

            "Image_ID":
                f"image_{image_index:03d}_jitter",

            "Original_File":
                str(image_path),

            "Output_File":
                str(output_path),

            "Original_Width":
                original_width,

            "Original_Height":
                original_height,

            "Output_Width":
                IMAGE_SIZE[0],

            "Output_Height":
                IMAGE_SIZE[1],

            "Transformation":
                "Resize + Random Crop + Color Jitter + Intensity Normalization"

        })


        total_image_outputs += 1


    except Exception as e:

        print(
            f"\nERROR processing image: "
            f"{image_path}"
        )

        print(
            "Error:",
            e
        )

image_metadata_df = pd.DataFrame(
    image_metadata
)


image_metadata_path = (
    IMAGE_OUTPUT /
    "Image_Preprocessing_Metadata.csv"
)


image_metadata_df.to_csv(
    image_metadata_path,
    index=False
)


print("\n")
print(
    "Image preprocessing completed."
)

print(
    "Original images processed :",
    len(image_files)
)

print(
    "Total output images       :",
    total_image_outputs
)

print(
    "Metadata saved to         :",
    image_metadata_path
)

print("\n")
print("=" * 75)
print("AUDIO PREPROCESSING")
print("=" * 75)

SAMPLE_RATE = 22050

N_MELS = 128

N_FFT = 2048

HOP_LENGTH = 512


audio_metadata = []

failed_audio = []


print(
    "\nSpectrogram output directory:"
)

print(
    SPECTROGRAM_OUTPUT
)

print(
    "\nActual audio files available:",
    len(audio_files)
)

for audio_index, audio_path in enumerate(

    tqdm(
        audio_files,
        desc="Processing audio"
    ),

    start=1

):

    audio_id = (
        f"audio_{audio_index:03d}"
    )


    print(
        f"\n[{audio_index}/{len(audio_files)}] "
        f"Processing: {audio_path.name}"
    )


    try:
        audio, sr = librosa.load(

            str(audio_path),

            sr=SAMPLE_RATE,

            mono=True

        )


        # MEL SPECTROGRAM

        mel_spectrogram = (
            librosa.feature.melspectrogram(

                y=audio,

                sr=sr,

                n_fft=N_FFT,

                hop_length=HOP_LENGTH,

                n_mels=N_MELS,

                power=2.0

            )
        )


        # LOGARITHMIC CONVERSION


        log_mel_spectrogram = (
            librosa.power_to_db(

                mel_spectrogram,

                ref=np.max

            )
        )


        # MIN-MAX NORMALIZATION

        minimum = (
            np.min(
                log_mel_spectrogram
            )
        )

        maximum = (
            np.max(
                log_mel_spectrogram
            )
        )


        if maximum > minimum:

            normalized_spectrogram = (

                log_mel_spectrogram
                - minimum

            ) / (

                maximum
                - minimum

            )

        else:

            normalized_spectrogram = (
                np.zeros_like(
                    log_mel_spectrogram
                )
            )


        normalized_spectrogram = np.clip(

            normalized_spectrogram,

            0.0,

            1.0

        )


        normalized_spectrogram = (
            normalized_spectrogram
            .astype(
                np.float32
            )
        )

        npy_path = (

            SPECTROGRAM_OUTPUT /
            f"{audio_id}_spectrogram.npy"

        )


        np.save(

            str(npy_path),

            normalized_spectrogram

        )

        png_path = (

            SPECTROGRAM_OUTPUT /
            f"{audio_id}_spectrogram.png"

        )


        fig = plt.figure(

            figsize=(
                10,
                5
            )

        )


        librosa.display.specshow(

            normalized_spectrogram,

            sr=sr,

            hop_length=HOP_LENGTH,

            x_axis="time",

            y_axis="mel"

        )


        plt.title(

            f"Log-Mel Spectrogram - {audio_id}",

            fontweight="bold"

        )


        plt.xlabel(

            "Time",

            fontweight="bold"

        )


        plt.ylabel(

            "Mel Frequency",

            fontweight="bold"

        )


        plt.colorbar()


        plt.tight_layout()


        fig.savefig(

            str(png_path),

            dpi=300,

            bbox_inches="tight"

        )


        plt.close(
            fig
        )

        duration = (
            len(audio) /
            sr
        )


        audio_metadata.append({

            "Audio_ID":
                audio_id,

            "Original_File":
                str(audio_path),

            "Original_File_Name":
                audio_path.name,

            "Spectrogram_NPY":
                str(npy_path),

            "Spectrogram_PNG":
                str(png_path),

            "Sample_Rate":
                sr,

            "Duration_Seconds":
                duration,

            "N_Mels":
                N_MELS,

            "N_FFT":
                N_FFT,

            "Hop_Length":
                HOP_LENGTH,

            "Spectrogram_Height":
                normalized_spectrogram.shape[0],

            "Spectrogram_Width":
                normalized_spectrogram.shape[1],

            "Processing":
                "Mel Spectrogram + Logarithmic Conversion + Min-Max Normalization"

        })


        print(
            f"    SUCCESS -> {audio_id}"
        )


    except Exception as e:

        failed_audio.append({

            "Audio_ID":
                audio_id,

            "Original_File":
                str(audio_path),

            "Error":
                str(e)

        })


        print(
            f"    FAILED -> {audio_id}"
        )

        print(
            "    Error:",
            e
        )

audio_metadata_df = pd.DataFrame(
    audio_metadata
)


audio_metadata_path = (
    AUDIO_OUTPUT /
    "Audio_Preprocessing_Metadata.csv"
)


audio_metadata_df.to_csv(

    audio_metadata_path,

    index=False

)

if len(failed_audio) > 0:

    failed_audio_path = (
        AUDIO_OUTPUT /
        "Failed_Audio_Files.csv"
    )

    pd.DataFrame(
        failed_audio
    ).to_csv(

        failed_audio_path,

        index=False

    )


print("\n")
print("=" * 75)
print("AUDIO PREPROCESSING SUMMARY")
print("=" * 75)

print(
    "Audio files found       :",
    len(audio_files)
)

print(
    "Successfully processed :",
    len(audio_metadata)
)

print(
    "Failed                  :",
    len(failed_audio)
)

print(
    "Metadata                :",
    audio_metadata_path
)

if len(failed_audio) == 0:

    print(
        "\nAll audio files processed successfully."
    )

print("\n")
print("=" * 75)
print("CSV PREPROCESSING")
print("=" * 75)

CSV_PATH = csv_files[0]


print(
    "\nUsing CSV:"
)

print(
    CSV_PATH
)

df = pd.read_csv(
    CSV_PATH
)


print(
    "\nRows    :",
    df.shape[0]
)

print(
    "Columns :",
    df.shape[1]
)

TARGET_COLUMN = (
    "Dissemination_Effectiveness"
)


if TARGET_COLUMN not in df.columns:

    raise ValueError(

        f"Target column "
        f"'{TARGET_COLUMN}' "
        f"not found in CSV."

    )


print("\n")
print("=" * 75)
print("TARGET INFORMATION")
print("=" * 75)


print(
    "\nTarget column:"
)

print(
    TARGET_COLUMN
)


print(
    "\nTarget distribution:"
)

print(
    df[
        TARGET_COLUMN
    ].value_counts()
)

text_columns = []


for column in df.columns:

    if column == TARGET_COLUMN:

        continue


    if df[column].dtype == "object":

        column_lower = (
            column.lower()
        )

        if any(

            keyword in column_lower

            for keyword in [

                "id",
                "label",
                "class",
                "target"

            ]

        ):

            continue


        text_columns.append(
            column
        )


print("\n")
print("=" * 75)
print("TEXT FEATURES")
print("=" * 75)


print(
    "\nDetected text columns:"
)

for column in text_columns:

    print(
        column
    )

def normalize_text(
    text
):

    if pd.isna(text):

        return ""


    text = str(
        text
    )


    # Lowercase
    text = text.lower()


    # Remove URLs
    text = re.sub(

        r"http\S+|www\S+",

        " ",

        text

    )


    text = re.sub(

        r"[^a-zA-Z0-9\s]",

        " ",

        text

    )

    text = re.sub(

        r"\s+",

        " ",

        text

    )


    return text.strip()

def tokenize_text(
    text
):

    normalized = (
        normalize_text(
            text
        )
    )


    tokens = (
        normalized.split()
    )


    return tokens


print("\n")
print("=" * 75)
print("TEXT NORMALIZATION + TOKENIZATION")
print("=" * 75)


for column in text_columns:

    normalized_column = (
        f"{column}_Normalized"
    )

    token_column = (
        f"{column}_Tokens"
    )


    df[
        normalized_column
    ] = (

        df[column]
        .apply(
            normalize_text
        )

    )


    df[
        token_column
    ] = (

        df[
            normalized_column
        ]

        .apply(
            tokenize_text
        )

        .apply(
            lambda x:
            " ".join(x)
        )

    )


    print(
        f"Processed: {column}"
    )

combined_text = []


for _, row in df.iterrows():

    text_parts = []


    for column in text_columns:

        normalized_column = (
            f"{column}_Normalized"
        )


        value = str(
            row[
                normalized_column
            ]
        )


        if value.strip():

            text_parts.append(
                value
            )


    combined_text.append(
        " ".join(
            text_parts
        )
    )


df[
    "Combined_Text"
] = combined_text


print(
    "\nCombined text generated."
)

print("\n")
print("=" * 75)
print("GENERATING TEXT EMBEDDINGS")
print("=" * 75)


print(
    "\nLoading Sentence Transformer..."
)


embedding_model = (
    SentenceTransformer(
        "all-MiniLM-L6-v2"
    )
)


embeddings = (
    embedding_model.encode(

        combined_text,

        batch_size=32,

        show_progress_bar=True,

        convert_to_numpy=True,

        normalize_embeddings=True

    )
)


print(
    "\nEmbedding shape:",
    embeddings.shape
)


# ============================================================
# SAVE EMBEDDINGS
# ============================================================

embedding_npy_path = (

    EMBEDDING_OUTPUT /
    "Text_Embeddings.npy"

)


np.save(

    str(embedding_npy_path),

    embeddings.astype(
        np.float32
    )

)


embedding_columns = [

    f"Embedding_{i+1}"

    for i in range(
        embeddings.shape[1]
    )

]


embedding_df = pd.DataFrame(

    embeddings,

    columns=embedding_columns

)


embedding_csv_path = (

    EMBEDDING_OUTPUT /
    "Text_Embeddings.csv"

)


embedding_df.to_csv(

    embedding_csv_path,

    index=False

)


print(
    "\nText embeddings generated successfully."
)

print(
    "Embedding dimension :",
    embeddings.shape[1]
)

print(
    "NPY file            :",
    embedding_npy_path
)

print(
    "CSV file            :",
    embedding_csv_path
)

numeric_columns = (
    df.select_dtypes(
        include=[
            np.number
        ]
    ).columns.tolist()
)


numeric_feature_columns = []


for column in numeric_columns:

    column_lower = (
        column.lower()
    )


    if any(

        keyword in column_lower

        for keyword in [

            "target",
            "label",
            "class"

        ]

    ):

        continue


    numeric_feature_columns.append(
        column
    )


print("\n")
print("=" * 75)
print("NUMERICAL FEATURES")
print("=" * 75)


print(
    "\nNumerical feature columns:"
)

for column in numeric_feature_columns:

    print(
        column
    )

print("\n")
print(
    "Applying Z-score normalization..."
)


if len(
    numeric_feature_columns
) > 0:


    scaler = (
        StandardScaler()
    )


    numeric_data = (

        df[
            numeric_feature_columns
        ]

        .replace(
            [np.inf, -np.inf],
            np.nan
        )

        .fillna(0)

    )


    normalized_numeric = (
        scaler.fit_transform(
            numeric_data
        )
    )


    normalized_numeric_df = (
        pd.DataFrame(

            normalized_numeric,

            columns=[
                f"{column}_Normalized"
                for column
                in numeric_feature_columns
            ],

            index=df.index

        )
    )


    # Add normalized features
    df = pd.concat(

        [
            df,
            normalized_numeric_df
        ],

        axis=1

    )


    print(
        "Numerical normalization completed."
    )

print("\n")
print("=" * 75)
print("TARGET RETENTION")
print("=" * 75)

df[
    "Target_Original"
] = (

    df[
        TARGET_COLUMN
    ]

)

target_encoder = (
    LabelEncoder()
)


target_encoded = (
    target_encoder.fit_transform(

        df[
            TARGET_COLUMN
        ].astype(
            str
        )

    )
)


df[
    "Target_Encoded"
] = (
    target_encoded
)


print(
    "\nTarget classes:"
)


for index, class_name in enumerate(
    target_encoder.classes_
):

    print(
        f"{index} -> {class_name}"
    )

target_mapping = {

    int(index):
        str(class_name)

    for index, class_name

    in enumerate(
        target_encoder.classes_
    )

}


target_mapping_path = (

    CSV_OUTPUT /
    "Target_Class_Mapping.json"

)


with open(

    target_mapping_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        target_mapping,

        f,

        indent=4,

        ensure_ascii=False

    )

df[
    "Embedding_Index"
] = np.arange(
    len(df)
)

processed_csv_path = (

    CSV_OUTPUT /
    "Preprocessed_Dataset_With_Target.csv"

)


df.to_csv(

    processed_csv_path,

    index=False

)


print(
    "\nProcessed CSV saved:"
)

print(
    processed_csv_path
)

target_df = pd.DataFrame({

    "Target_Original":
        df[
            TARGET_COLUMN
        ],

    "Target_Encoded":
        df[
            "Target_Encoded"
        ]

})


target_path = (

    CSV_OUTPUT /
    "Target.csv"

)


target_df.to_csv(

    target_path,

    index=False

)


print(
    "\nTarget file saved:"
)

print(
    target_path
)

if len(
    numeric_feature_columns
) > 0:

    normalized_numeric_path = (

        CSV_OUTPUT /
        "Normalized_Numerical_Features.csv"

    )


    normalized_numeric_df.to_csv(

        normalized_numeric_path,

        index=False

    )


    print(
        "\nNormalized numerical features saved:"
    )

    print(
        normalized_numeric_path
    )

text_feature_columns = []


for column in text_columns:

    text_feature_columns.append(
        column
    )

    text_feature_columns.append(
        f"{column}_Normalized"
    )

    text_feature_columns.append(
        f"{column}_Tokens"
    )


text_features_path = (

    CSV_OUTPUT /
    "Processed_Text_Features.csv"

)


df[
    text_feature_columns
].to_csv(

    text_features_path,

    index=False

)


print(
    "\nProcessed text features saved:"
)

print(
    text_features_path
)

summary = {

    "Project":
        "Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage",

    "Dataset":
        str(CSV_PATH),

    "Rows":
        int(
            len(df)
        ),

    "Original_Columns":
        38,

    "Final_Columns":
        int(
            len(df.columns)
        ),

    "Image_Files":
        int(
            len(image_files)
        ),

    "Image_Output_Count":
        int(
            total_image_outputs
        ),

    "Audio_Files":
        int(
            len(audio_files)
        ),

    "Audio_Successfully_Processed":
        int(
            len(audio_metadata)
        ),

    "Audio_Failed":
        int(
            len(failed_audio)
        ),

    "Target":
        TARGET_COLUMN,

    "Target_Preserved":
        True,

    "Target_Used_In_Text_Embedding":
        False,

    "Text_Input_Columns":
        text_columns,

    "Text_Embedding_Model":
        "all-MiniLM-L6-v2",

    "Text_Embedding_Dimension":
        int(
            embeddings.shape[1]
        ),

    "Text_Embedding_Shape":
        [
            int(
                embeddings.shape[0]
            ),
            int(
                embeddings.shape[1]
            )
        ],

    "Numerical_Features":
        numeric_feature_columns,

    "Numerical_Normalization":
        "Z-score StandardScaler",

    "Target_Encoding":
        "LabelEncoder",

    "Target_Classes":
        [
            str(x)
            for x
            in target_encoder.classes_
        ],

    "Image_Processing":
        [
            "Resize",
            "Intensity Normalization",
            "Random Crop",
            "Horizontal Flip",
            "Color Jitter"
        ],

    "Audio_Processing":
        [
            "Mel Spectrogram",
            "Logarithmic Conversion",
            "Min-Max Normalization"
        ],

    "Text_Processing":
        [
            "Text Normalization",
            "Tokenization",
            "Sentence-BERT Embedding"
        ]

}


summary_path = (

    OUTPUT_PATH /
    "Preprocessing_Summary.json"

)


with open(

    summary_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=4,

        ensure_ascii=False

    )

print("\n")
print("=" * 75)
print("FINAL PREPROCESSING VALIDATION")
print("=" * 75)


print(
    "\nImages found              :",
    len(image_files)
)

print(
    "Image outputs             :",
    total_image_outputs
)

print(
    "Audio files found         :",
    len(audio_files)
)

print(
    "Audio spectrograms        :",
    len(audio_metadata)
)

print(
    "Audio failures            :",
    len(failed_audio)
)

print(
    "CSV rows                  :",
    len(df)
)

print(
    "CSV original columns     :",
    38
)

print(
    "CSV final columns        :",
    len(df.columns)
)

print(
    "Text embedding shape     :",
    embeddings.shape
)

print(
    "Target column            :",
    TARGET_COLUMN
)

print(
    "Target preserved         :",
    TARGET_COLUMN in df.columns
)

print(
    "Target encoded           :",
    "Target_Encoded" in df.columns
)

print(
    "Target used in embedding :",
    False
)


print("\nTarget distribution:")

print(
    df[
        TARGET_COLUMN
    ].value_counts()
)

print("\n")
print("=" * 75)
print("GENERATED OUTPUT")
print("=" * 75)


print(
    "\nImages:"
)

print(
    IMAGE_OUTPUT
)


print(
    "\nAudios:"
)

print(
    AUDIO_OUTPUT
)


print(
    "\nSpectrograms:"
)

print(
    SPECTROGRAM_OUTPUT
)


print(
    "\nCSV:"
)

print(
    CSV_OUTPUT
)


print(
    "\nEmbeddings:"
)

print(
    EMBEDDING_OUTPUT
)


print(
    "\nProcessed dataset:"
)

print(
    processed_csv_path
)


print(
    "\nTarget:"
)

print(
    target_path
)


print(
    "\nSummary:"
)

print(
    summary_path
)

print("\n")
print("=" * 75)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    "\nImages:",
    len(image_files),
    "original →",
    total_image_outputs,
    "processed"
)

print(
    "Audio:",
    len(audio_files),
    "found →",
    len(audio_metadata),
    "spectrograms"
)

print(
    "CSV:",
    len(df),
    "rows"
)

print(
    "Text embeddings:",
    embeddings.shape
)

print(
    "Target:",
    TARGET_COLUMN,
    "preserved"
)

print(
    "\nAll preprocessing outputs are available at:"
)

print(
    OUTPUT_PATH
)

print("\n" + "=" * 75)




MULTIMODAL DATA PREPROCESSING

Dataset path:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Dataset

Output path:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output


DATASET FILE SUMMARY

Images found : 50
Audios found : 6
CSV files    : 1


IMAGE PREPROCESSING


Processing images: 100%|███████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 25.11it/s]




Image preprocessing completed.
Original images processed : 50
Total output images       : 200
Metadata saved to         : D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output\Images\Image_Preprocessing_Metadata.csv


AUDIO PREPROCESSING

Spectrogram output directory:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output\Audios\Spectrograms

Actual audio files available: 6


Processing audio:   0%|                                                                          | 0/6 [00:00<?, ?it/s]


[1/6] Processing: 1.mp3


...

Processing audio: 100%|██████████████████████████████████████████████████████████████████| 6/6 [01:35<00:00, 15.94s/it]

    SUCCESS -> audio_006




AUDIO PREPROCESSING SUMMARY
Audio files found       : 6
Successfully processed : 6
Failed                  : 0
Metadata                : D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output\Audios\Audio_Preprocessing_Metadata.csv

All audio files processed successfully.


CSV PREPROCESSING

Using CSV:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Dataset\Cultural_Heritage_Dissemination_Dataset.csv

Rows    : 7000
Columns : 38


TARGET INFORMATION

Target column:
Dissemination_Effectiveness

Target distribution:
Dissemination_Effectiveness
Very High    1400
Moderate     1400
Low          1400
Very Low     1400
High         1400
Name: count, dtype: int64


TEXT FEATURES

Detected text columns:
Heritage_Category
Content_Language
Region_Type
Text_Con

Batches:   0%|          | 0/219 [00:00<?, ?it/s]


Embedding shape: (7000, 384)

Text embeddings generated successfully.
Embedding dimension : 384
NPY file            : D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output\CSV\Embeddings\Text_Embeddings.npy
CSV file            : D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output\CSV\Embeddings\Text_Embeddings.csv


NUMERICAL FEATURES

Numerical feature columns:
Content_ID
Visual_Quality
Visual_Clarity
Visual_Detail
Color_Richness
Text_Quality
Text_Completeness
Cultural_Context_Score
Description_Length
Keyword_Relevance
Audio_Quality
Audio_Clarity
Audio_Duration_Seconds
Metadata_Completeness
Source_Reliability
Cultural_Authenticity
Historical_Information
User_Age
Previous_Cultural_Interest
Content_Interest
Session_Duration_Minu

# Feature Extraction

In [4]:
import sys
import subprocess
import importlib
import warnings
import json
import re

from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

from tqdm import tqdm

import tensorflow as tf

from tensorflow.keras import (
    layers,
    models
)

warnings.filterwarnings("ignore")

def install_if_missing(
    package_name,
    import_name=None
):

    if import_name is None:
        import_name = package_name

    try:
        importlib.import_module(
            import_name
        )

    except ImportError:

        print(
            f"Installing {package_name}..."
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package_name
            ]
        )


install_if_missing("numpy")
install_if_missing("pandas")
install_if_missing("Pillow", "PIL")
install_if_missing("tqdm")
install_if_missing("tensorflow")

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

PREPROCESSED_PATH = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output"
)

IMAGE_PATH = (
    PREPROCESSED_PATH /
    "Images"
)

AUDIO_PATH = (
    PREPROCESSED_PATH /
    "Audios" /
    "Spectrograms"
)

CSV_PATH = (
    PREPROCESSED_PATH /
    "CSV"
)

EMBEDDING_PATH = (
    CSV_PATH /
    "Embeddings"
)

FEATURE_OUTPUT = (
    PREPROCESSED_PATH.parent /
    "Feature Extraction"
)

IMAGE_FEATURE_OUTPUT = (
    FEATURE_OUTPUT /
    "Image_3D_CNN"
)

AUDIO_FEATURE_OUTPUT = (
    FEATURE_OUTPUT /
    "Audio_Spectrogram_CNN"
)

TEXT_FEATURE_OUTPUT = (
    FEATURE_OUTPUT /
    "Text_Transformer"
)

CSV_FEATURE_OUTPUT = (
    FEATURE_OUTPUT /
    "CSV_Features"
)

FINAL_FEATURE_OUTPUT = (
    FEATURE_OUTPUT /
    "Multimodal"
)


for folder in [

    FEATURE_OUTPUT,
    IMAGE_FEATURE_OUTPUT,
    AUDIO_FEATURE_OUTPUT,
    TEXT_FEATURE_OUTPUT,
    CSV_FEATURE_OUTPUT,
    FINAL_FEATURE_OUTPUT

]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

IMAGE_SIZE = 224

IMAGE_CHANNELS = 3

IMAGE_AUGMENTATIONS = 4

IMAGE_FEATURE_DIM = 256

AUDIO_HEIGHT = 128

AUDIO_WIDTH = 256

AUDIO_FEATURE_DIM = 256

TEXT_FEATURE_DIM = 256

CSV_FEATURE_DIM = 128

BATCH_SIZE = 16

TARGET_COLUMN = (
    "Dissemination_Effectiveness"
)

print("\n")
print("=" * 80)
print("MULTIMODAL DEEP FEATURE EXTRACTION")
print("=" * 80)

print("\nPreprocessed data:")
print(PREPROCESSED_PATH)

print("\nFeature output:")
print(FEATURE_OUTPUT)

for required_path in [

    IMAGE_PATH,
    AUDIO_PATH,
    CSV_PATH,
    EMBEDDING_PATH

]:

    if not required_path.exists():

        raise FileNotFoundError(
            f"Required path not found:\n{required_path}"
        )

print("\n")
print("=" * 80)
print("COMPUTATION DEVICE")
print("=" * 80)

gpus = tf.config.list_physical_devices(
    "GPU"
)

if gpus:

    print(
        "GPU detected:",
        gpus
    )

else:

    print(
        "GPU not detected."
    )

    print(
        "TensorFlow will use CPU."
    )

# VISUAL FEATURE EXTRACTION USING 3D CNN

print("\n")
print("=" * 80)
print("1. VISUAL FEATURE EXTRACTION USING CNN (3D CNN)")
print("=" * 80)

image_files = sorted(

    [

        p

        for p in IMAGE_PATH.iterdir()

        if p.suffix.lower()
        in {
            ".png",
            ".jpg",
            ".jpeg"
        }

    ],

    key=lambda x: x.name.lower()

)


print(
    "\nPreprocessed image files:",
    len(image_files)
)


if len(image_files) == 0:

    raise RuntimeError(
        "No preprocessed images found."
    )

def load_image(
    path
):

    image = Image.open(
        path
    ).convert(
        "RGB"
    )

    image = image.resize(

        (
            IMAGE_SIZE,
            IMAGE_SIZE
        ),

        Image.Resampling.BILINEAR

    )

    image = np.asarray(
        image
    ).astype(
        np.float32
    )

    image /= 255.0

    return image

def build_3d_cnn():

    input_layer = layers.Input(

        shape=(

            IMAGE_AUGMENTATIONS,
            IMAGE_SIZE,
            IMAGE_SIZE,
            IMAGE_CHANNELS

        ),

        name="Image_3D_Input"

    )

    x = layers.Conv3D(

        32,
        kernel_size=(3, 3, 3),
        padding="same",
        activation="relu"

    )(input_layer)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling3D(

        pool_size=(1, 2, 2)

    )(x)

    x = layers.Conv3D(

        64,
        kernel_size=(3, 3, 3),
        padding="same",
        activation="relu"

    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling3D(

        pool_size=(2, 2, 2)

    )(x)

    x = layers.Conv3D(

        128,
        kernel_size=(3, 3, 3),
        padding="same",
        activation="relu"

    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling3D(

        pool_size=(1, 2, 2)

    )(x)

    x = layers.Conv3D(

        256,
        kernel_size=(3, 3, 3),
        padding="same",
        activation="relu"

    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling3D()(x)

    features = layers.Dense(

        IMAGE_FEATURE_DIM,
        activation="relu",
        name="Visual_Feature_Vector"

    )(x)

    model = models.Model(

        input_layer,
        features,
        name="Visual_3D_CNN"

    )

    return model


visual_model = build_3d_cnn()

visual_model.summary()

print("\n")
print(
    "Grouping images into 3D CNN samples..."
)


image_groups = []

for image_number in range(

    1,

    (len(image_files) // IMAGE_AUGMENTATIONS) + 1

):

    group = []

    suffixes = [

        "original",
        "crop",
        "flip",
        "jitter"

    ]

    valid = True

    for suffix in suffixes:

        filename = (

            f"image_{image_number:03d}_"
            f"{suffix}.png"

        )

        path = IMAGE_PATH / filename

        if not path.exists():

            valid = False

            break

        group.append(
            path
        )

    if valid:

        image_groups.append(
            group
        )


print(
    "3D CNN image groups:",
    len(image_groups)
)


if len(image_groups) == 0:

    raise RuntimeError(
        "No complete image groups found."
    )

visual_features = []

visual_ids = []


for group_index, group in enumerate(

    tqdm(
        image_groups,
        desc="Extracting visual features"
    ),

    start=1

):

    volume = [

        load_image(
            image_path
        )

        for image_path in group

    ]

    volume = np.stack(

        volume,

        axis=0

    )

    volume = np.expand_dims(

        volume,

        axis=0

    )

    features = visual_model.predict(

        volume,

        verbose=0

    )

    visual_features.append(
        features[0]
    )

    visual_ids.append(

        f"image_{group_index:03d}"

    )


visual_features = np.asarray(

    visual_features,

    dtype=np.float32

)


print(
    "\nVisual feature shape:",
    visual_features.shape
)

np.save(

    IMAGE_FEATURE_OUTPUT /
    "Visual_3D_CNN_Features.npy",

    visual_features

)


visual_df = pd.DataFrame(

    visual_features,

    columns=[

        f"Visual_Feature_{i+1}"

        for i in range(
            visual_features.shape[1]
        )

    ]

)


visual_df.insert(

    0,

    "Image_ID",

    visual_ids

)


visual_df.to_csv(

    IMAGE_FEATURE_OUTPUT /
    "Visual_3D_CNN_Features.csv",

    index=False

)

# AUDIO FEATURE EXTRACTION

print("\n")
print("=" * 80)
print("2. AUDIO FEATURE EXTRACTION USING SPECTROGRAM-BASED CNN")
print("=" * 80)

spectrogram_files = sorted(

    [

        p

        for p in AUDIO_PATH.iterdir()

        if p.suffix.lower() == ".npy"

    ],

    key=lambda x: x.name.lower()

)


print(
    "\nSpectrogram files:",
    len(spectrogram_files)
)


if len(spectrogram_files) == 0:

    raise RuntimeError(
        "No spectrogram files found."
    )

def load_spectrogram(
    path
):

    spec = np.load(
        path
    ).astype(
        np.float32
    )

    spec = np.nan_to_num(
        spec
    )

    minimum = np.min(
        spec
    )

    maximum = np.max(
        spec
    )

    if maximum > minimum:

        spec = (

            spec - minimum

        ) / (

            maximum - minimum

        )

    else:

        spec = np.zeros_like(
            spec
        )

    spec_uint8 = (

        spec * 255

    ).astype(
        np.uint8
    )

    image = Image.fromarray(
        spec_uint8
    )

    image = image.resize(

        (
            AUDIO_WIDTH,
            AUDIO_HEIGHT
        ),

        Image.Resampling.BILINEAR

    )

    spec = (

        np.asarray(
            image
        ).astype(
            np.float32
        )
        / 255.0

    )

    spec = np.expand_dims(

        spec,

        axis=-1

    )

    return spec

def build_audio_cnn():

    input_layer = layers.Input(

        shape=(

            AUDIO_HEIGHT,
            AUDIO_WIDTH,
            1

        ),

        name="Spectrogram_Input"

    )

    x = layers.Conv2D(

        32,
        3,
        padding="same",
        activation="relu"

    )(input_layer)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(

        64,
        3,
        padding="same",
        activation="relu"

    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(

        128,
        3,
        padding="same",
        activation="relu"

    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(

        256,
        3,
        padding="same",
        activation="relu"

    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)

    features = layers.Dense(

        AUDIO_FEATURE_DIM,
        activation="relu",
        name="Audio_Feature_Vector"

    )(x)

    model = models.Model(

        input_layer,
        features,
        name="Audio_Spectrogram_CNN"

    )

    return model


audio_model = build_audio_cnn()

audio_model.summary()

audio_features = []

audio_ids = []


for audio_path in tqdm(

    spectrogram_files,

    desc="Extracting audio features"

):

    try:

        spec = load_spectrogram(
            audio_path
        )

        spec = np.expand_dims(
            spec,
            axis=0
        )

        features = audio_model.predict(

            spec,

            verbose=0

        )

        audio_features.append(
            features[0]
        )

        audio_ids.append(

            audio_path.stem.replace(
                "_spectrogram",
                ""
            )

        )

    except Exception as e:

        print(
            f"\nERROR: {audio_path.name}"
        )

        print(e)


audio_features = np.asarray(

    audio_features,

    dtype=np.float32

)


print(
    "\nAudio feature shape:",
    audio_features.shape
)

np.save(

    AUDIO_FEATURE_OUTPUT /
    "Audio_Spectrogram_CNN_Features.npy",

    audio_features

)


audio_df = pd.DataFrame(

    audio_features,

    columns=[

        f"Audio_Feature_{i+1}"

        for i in range(
            audio_features.shape[1]
        )

    ]

)


audio_df.insert(

    0,

    "Audio_ID",

    audio_ids

)


audio_df.to_csv(

    AUDIO_FEATURE_OUTPUT /
    "Audio_Spectrogram_CNN_Features.csv",

    index=False

)

# TEXTUAL SEMANTIC ENCODING USING TRANSFORMER

print("\n")
print("=" * 80)
print("3. TEXTUAL SEMANTIC ENCODING USING TRANSFORMER NETWORKS")
print("=" * 80)

embedding_npy = (

    EMBEDDING_PATH /
    "Text_Embeddings.npy"

)


if not embedding_npy.exists():

    raise FileNotFoundError(
        f"Text embedding file not found:\n{embedding_npy}"
    )


text_embeddings = np.load(
    embedding_npy
).astype(
    np.float32
)


text_embeddings = np.nan_to_num(
    text_embeddings
)


print(
    "\nInput text embedding shape:",
    text_embeddings.shape
)

TEXT_INPUT_DIM = (
    text_embeddings.shape[1]
)


def build_text_transformer():

    input_layer = layers.Input(

        shape=(
            TEXT_INPUT_DIM,
        ),

        name="Transformer_Text_Input"

    )

    x = layers.Dense(

        256,
        activation="relu"

    )(input_layer)

    x = layers.Reshape(

        (16, 16)

    )(x)


    positions = tf.range(

        start=0,
        limit=16,
        delta=1

    )


    position_embedding = layers.Embedding(

        input_dim=16,
        output_dim=16

    )(positions)


    x = x + position_embedding


    attention_output = layers.MultiHeadAttention(

        num_heads=4,
        key_dim=16,
        dropout=0.10

    )(

        x,
        x

    )


    x = layers.Add()(
        [
            x,
            attention_output
        ]
    )


    x = layers.LayerNormalization()(x)


    ff = layers.Dense(

        128,
        activation="relu"

    )(x)


    ff = layers.Dense(

        16

    )(ff)


    x = layers.Add()(
        [
            x,
            ff
        ]
    )


    x = layers.LayerNormalization()(x)


    x = layers.GlobalAveragePooling1D()(x)


    features = layers.Dense(

        TEXT_FEATURE_DIM,
        activation="relu",
        name="Transformer_Semantic_Features"

    )(x)


    model = models.Model(

        input_layer,
        features,

        name="Textual_Semantic_Transformer"

    )


    return model


text_model = build_text_transformer()

text_model.summary()

text_features = text_model.predict(

    text_embeddings,

    batch_size=BATCH_SIZE,

    verbose=1

)


text_features = np.asarray(

    text_features,

    dtype=np.float32

)


print(
    "\nText semantic feature shape:",
    text_features.shape
)

np.save(

    TEXT_FEATURE_OUTPUT /
    "Text_Transformer_Semantic_Features.npy",

    text_features

)


text_df = pd.DataFrame(

    text_features,

    columns=[

        f"Text_Feature_{i+1}"

        for i in range(
            text_features.shape[1]
        )

    ]

)


text_df.insert(

    0,

    "Text_Index",

    np.arange(
        len(text_df)
    )

)


text_df.to_csv(

    TEXT_FEATURE_OUTPUT /
    "Text_Transformer_Semantic_Features.csv",

    index=False

)

# CORRECTED CSV NUMERICAL FEATURE EXTRACTION

print("\n")
print("=" * 80)
print("4. CSV NUMERICAL FEATURE EXTRACTION")
print("=" * 80)

possible_csv_files = [

    CSV_PATH /
    "Preprocessed_Dataset_With_Target.csv",

    CSV_PATH /
    "Preprocessed_Dataset.csv"

]


processed_csv = None


for candidate in possible_csv_files:

    if candidate.exists():

        processed_csv = candidate

        break


if processed_csv is None:

    raise FileNotFoundError(

        "Could not find preprocessed CSV.\n"
        "Checked:\n" +
        "\n".join(
            str(x)
            for x in possible_csv_files
        )

    )


print(
    "\nUsing CSV:"
)

print(
    processed_csv
)

df = pd.read_csv(
    processed_csv
)


print(
    "\nCSV shape:",
    df.shape
)

if TARGET_COLUMN not in df.columns:

    raise ValueError(

        f"Target column '{TARGET_COLUMN}' "
        "was not found in CSV."

    )


print(
    "\nTarget column:",
    TARGET_COLUMN
)


print(
    "\nTarget distribution:"
)

print(
    df[TARGET_COLUMN].value_counts()
)

numeric_columns = (

    df.select_dtypes(
        include=[
            np.number
        ]
    ).columns.tolist()

)


print(
    "\nAll numeric columns:"
)

print(
    numeric_columns
)

excluded_columns = {

    TARGET_COLUMN,

    "Target_Encoded"

}

for column in df.columns:

    lower_column = column.lower()

    if (

        "target" in lower_column

        and
        column != "Content_ID"

    ):

        excluded_columns.add(
            column
        )


numeric_feature_columns = [

    column

    for column in numeric_columns

    if column not in excluded_columns

]


print(
    "\nExcluded target-related columns:"
)

print(
    sorted(
        excluded_columns
    )
)


print(
    "\nNumeric feature columns used:"
)

for column in numeric_feature_columns:

    print(
        " -",
        column
    )


print(
    "\nNumber of numeric input features:",
    len(numeric_feature_columns)
)


if len(numeric_feature_columns) == 0:

    raise RuntimeError(
        "No numeric feature columns available."
    )

numeric_features = (

    df[
        numeric_feature_columns
    ]

    .apply(
        pd.to_numeric,
        errors="coerce"
    )

    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )

    .fillna(0)

    .values

    .astype(
        np.float32
    )

)


print(
    "\nNumerical input shape:",
    numeric_features.shape
)

if not np.isfinite(
    numeric_features
).all():

    raise ValueError(
        "Numeric feature matrix still contains "
        "NaN or infinite values."
    )

CSV_INPUT_DIM = (
    numeric_features.shape[1]
)


csv_input = layers.Input(

    shape=(
        CSV_INPUT_DIM,
    ),

    name="CSV_Numerical_Input"

)


x = layers.Dense(

    256,
    activation="relu"

)(csv_input)


x = layers.BatchNormalization()(x)


x = layers.Dropout(
    0.15
)(x)


x = layers.Dense(

    CSV_FEATURE_DIM,
    activation="relu",
    name="CSV_Feature_Vector"

)(x)


csv_model = models.Model(

    csv_input,
    x,

    name="CSV_Deep_Feature_Extractor"

)


csv_model.summary()

csv_features = csv_model.predict(

    numeric_features,

    batch_size=BATCH_SIZE,

    verbose=1

)


csv_features = np.asarray(

    csv_features,

    dtype=np.float32

)


print(
    "\nCSV feature shape:",
    csv_features.shape
)

np.save(

    CSV_FEATURE_OUTPUT /
    "CSV_Numerical_Deep_Features.npy",

    csv_features

)


csv_features_df = pd.DataFrame(

    csv_features,

    columns=[

        f"CSV_Feature_{i+1}"

        for i in range(
            csv_features.shape[1]
        )

    ]

)


if "Content_ID" in df.columns:

    csv_features_df.insert(

        0,

        "Content_ID",

        df["Content_ID"].values

    )


csv_features_df.to_csv(

    CSV_FEATURE_OUTPUT /
    "CSV_Numerical_Deep_Features.csv",

    index=False

)

# TARGET FILE

print("\n")
print("=" * 80)
print("TARGET PREPARATION")
print("=" * 80)


target_df = pd.DataFrame()


if "Content_ID" in df.columns:

    target_df["Content_ID"] = (
        df["Content_ID"].values
    )


target_df[
    "Dissemination_Effectiveness"
] = (

    df[
        TARGET_COLUMN
    ].values

)

if "Target_Encoded" in df.columns:

    target_df[
        "Target_Encoded"
    ] = (

        df[
            "Target_Encoded"
        ].values

    )


target_path = (

    FINAL_FEATURE_OUTPUT /
    "Target.csv"

)


target_df.to_csv(

    target_path,

    index=False

)

print(
    "\nTarget saved:"
)

print(
    target_path
)

print("\n")
print("=" * 80)
print("TABULAR + TEXT FEATURE FUSION")
print("=" * 80)


if (

    len(text_features)
    != len(df)

):

    raise ValueError(

        "Text feature count does not match CSV rows.\n"
        f"Text: {len(text_features)}\n"
        f"CSV : {len(df)}"

    )


if (

    len(csv_features)
    != len(df)

):

    raise ValueError(

        "CSV feature count does not match CSV rows.\n"
        f"CSV features: {len(csv_features)}\n"
        f"CSV rows    : {len(df)}"

    )

text_fusion_df = pd.DataFrame(

    text_features,

    columns=[

        f"Text_Feature_{i+1}"

        for i in range(
            text_features.shape[1]
        )

    ]

)

csv_fusion_df = pd.DataFrame(

    csv_features,

    columns=[

        f"CSV_Feature_{i+1}"

        for i in range(
            csv_features.shape[1]
        )

    ]

)

fusion_df = pd.concat(

    [

        df[
            [
                "Content_ID"
            ]
        ].reset_index(
            drop=True
        ),

        text_fusion_df.reset_index(
            drop=True
        ),

        csv_fusion_df.reset_index(
            drop=True
        )

    ],

    axis=1

)

fusion_df[
    TARGET_COLUMN
] = df[
    TARGET_COLUMN
].values


if "Target_Encoded" in df.columns:

    fusion_df[
        "Target_Encoded"
    ] = df[
        "Target_Encoded"
    ].values

fusion_csv_path = (

    FINAL_FEATURE_OUTPUT /
    "Text_CSV_Fused_Features_7000.csv"

)


fusion_df.to_csv(

    fusion_csv_path,

    index=False

)


print(
    "\nText + CSV fused feature shape:"
)

print(
    fusion_df.shape
)


print(
    "\nSaved:"
)

print(
    fusion_csv_path
)

print("\n")
print("=" * 80)
print("7. TARGET VALIDATION")
print("=" * 80)


print(
    "\nTarget preserved:",
    TARGET_COLUMN in fusion_df.columns
)


print(
    "\nTarget distribution in fused file:"
)

print(
    fusion_df[
        TARGET_COLUMN
    ].value_counts()
)

summary = {

    "Project":
        "Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage",

    "Preprocessed_Path":
        str(PREPROCESSED_PATH),

    "Feature_Output":
        str(FEATURE_OUTPUT),

    "Image_Preprocessed_Count":
        int(len(image_files)),

    "Image_3D_CNN_Samples":
        int(len(image_groups)),

    "Image_Feature_Dimension":
        int(visual_features.shape[1]),

    "Image_Feature_Shape":
        list(visual_features.shape),

    "Audio_Spectrogram_Count":
        int(len(spectrogram_files)),

    "Audio_Feature_Dimension":
        int(audio_features.shape[1]),

    "Audio_Feature_Shape":
        list(audio_features.shape),

    "Text_Embedding_Shape":
        list(text_embeddings.shape),

    "Text_Transformer_Feature_Dimension":
        int(text_features.shape[1]),

    "Text_Transformer_Feature_Shape":
        list(text_features.shape),

    "CSV_Original_Rows":
        int(len(df)),

    "CSV_Numeric_Input_Features":
        int(len(numeric_feature_columns)),

    "CSV_Deep_Feature_Dimension":
        int(csv_features.shape[1]),

    "CSV_Deep_Feature_Shape":
        list(csv_features.shape),

    "Text_CSV_Fused_Shape":
        list(fusion_df.shape),

    "Target":
        TARGET_COLUMN,

    "Target_Preserved":
        bool(
            TARGET_COLUMN
            in fusion_df.columns
        ),

    "Visual_Method":
        "Visual Feature Extraction Using CNN (3D CNN)",

    "Audio_Method":
        "Audio Feature Extraction Using Spectrogram-Based CNN",

    "Text_Method":
        "Textual Semantic Encoding Using Transformer Networks",

    "CSV_Method":
        "Deep Numerical Feature Extraction",

    "Image_Audio_Included_In_7000_Row_Fusion":
        False,

    "Reason":
        "Image and audio samples do not currently have a verified one-to-one Content_ID mapping."

}


summary_path = (

    FEATURE_OUTPUT /
    "Feature_Extraction_Summary.json"

)


with open(

    summary_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=4

    )

print("\n")
print("=" * 80)
print("FEATURE EXTRACTION COMPLETED SUCCESSFULLY")
print("=" * 80)


print("\nIMAGE")
print("-" * 50)

print(
    "Input images       :",
    len(image_files)
)

print(
    "3D CNN samples      :",
    len(image_groups)
)

print(
    "Visual features     :",
    visual_features.shape
)


print("\nAUDIO")
print("-" * 50)

print(
    "Input spectrograms  :",
    len(spectrogram_files)
)

print(
    "Audio features      :",
    audio_features.shape
)


print("\nTEXT")
print("-" * 50)

print(
    "Text embeddings     :",
    text_embeddings.shape
)

print(
    "Transformer features:",
    text_features.shape
)


print("\nCSV")
print("-" * 50)

print(
    "CSV rows            :",
    len(df)
)

print(
    "Numeric inputs      :",
    len(numeric_feature_columns)
)

print(
    "CSV deep features   :",
    csv_features.shape
)


print("\nFUSION")
print("-" * 50)

print(
    "Text + CSV fusion   :",
    fusion_df.shape
)

print(
    "Target              :",
    TARGET_COLUMN
)

print(
    "Target preserved    :",
    TARGET_COLUMN in fusion_df.columns
)


print("\nOUTPUT")
print("-" * 50)

print(
    FEATURE_OUTPUT
)


print("\nMain fused CSV:")
print(
    fusion_csv_path
)


print("\nSummary:")
print(
    summary_path
)


print("\n")
print("=" * 80)
print("ALL FEATURE EXTRACTION STAGES COMPLETED")
print("=" * 80)



MULTIMODAL DEEP FEATURE EXTRACTION

Preprocessed data:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output

Feature output:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Feature Extraction


COMPUTATION DEVICE
GPU not detected.
TensorFlow will use CPU.


1. VISUAL FEATURE EXTRACTION USING CNN (3D CNN)

Preprocessed image files: 200


Model: "Visual_3D_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Image_3D_Input (InputLayer)          │ (None, 4, 224, 224, 3)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d (Conv3D)                      │ (None, 4, 224, 224, 32)     │           2,624 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 4, 224, 224, 32)     │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d (MaxPooling3D)         │ (None, 4, 112, 112, 32)     │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_1 (Conv3D)                    │ (None, 4, 112, 112, 64)     │          55,360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 4, 112, 112, 64)     │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_1 (MaxPooling3D)       │ (None, 2, 56, 56, 64)       │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_2 (Conv3D)                    │ (None, 2, 56, 56, 128)      │         221,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 2, 56, 56, 128)      │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling3d_2 (MaxPooling3D)       │ (None, 2, 28, 28, 128)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3d_3 (Conv3D)                    │ (None, 2, 28, 28, 256)      │         884,992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 2, 28, 28, 256)      │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling3d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling3D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Visual_Feature_Vector (Dense)        │ (None, 256)                 │          65,792 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,232,000 (4.70 MB)

 Trainable params: 1,231,040 (4.70 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ CSV_Numerical_Input (InputLayer)     │ (None, 61)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 256)                 │          15,872 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_8                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ CSV_Feature_Vector (Dense)           │ (None, 128)                 │          32,896 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 49,792 (194.50 KB)

 Trainable params: 49,280 (192.50 KB)

 Non-trainable params: 512 (2.00 KB)

438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 977us/step

CSV feature shape: (7000, 128)


TARGET PREPARATION

Target saved:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Feature Extraction\Multimodal\Target.csv


TABULAR + TEXT FEATURE FUSION

Text + CSV fused feature shape:
(7000, 387)

Saved:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Feature Extraction\Multimodal\Text_CSV_Fused_Features_7000.csv


7. TARGET VALIDATION

Target preserved: True

Target distribution in fused file:
Dissemination_Effectiveness
Very High    1400
Moderate     1400
Low          1400
Very Low     1400
High         1400
Name: count, dtype: int64


FEATURE EXTRACTION COMPLETED SUCCESSFULLY

IMAGE
--------------------------------------------------
Input images       : 200
3D CNN samples    

# Multimodal Feature Fusion

In [5]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

FEATURE_ROOT = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Feature Extraction"
)

PREPROCESSED_ROOT = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Preprocessed Output"
)

FINAL_OUTPUT = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output"
)

FINAL_OUTPUT.mkdir(parents=True, exist_ok=True)

IMAGE_FEATURE_FILE = (
    FEATURE_ROOT
    / "Image_3D_CNN"
    / "Visual_3D_CNN_Features.npy"
)

IMAGE_FEATURE_CSV = (
    FEATURE_ROOT
    / "Image_3D_CNN"
    / "Visual_3D_CNN_Features.csv"
)

AUDIO_FEATURE_FILE = (
    FEATURE_ROOT
    / "Audio_Spectrogram_CNN"
    / "Audio_Spectrogram_CNN_Features.npy"
)

AUDIO_FEATURE_CSV = (
    FEATURE_ROOT
    / "Audio_Spectrogram_CNN"
    / "Audio_Spectrogram_CNN_Features.csv"
)

TEXT_FEATURE_FILE = (
    FEATURE_ROOT
    / "Text_Transformer"
    / "Text_Transformer_Semantic_Features.npy"
)

TEXT_FEATURE_CSV = (
    FEATURE_ROOT
    / "Text_Transformer"
    / "Text_Transformer_Semantic_Features.csv"
)

CSV_FEATURE_FILE = (
    FEATURE_ROOT
    / "CSV_Deep_Features"
    / "CSV_Deep_Features.csv"
)

TARGET_FILE = (
    PREPROCESSED_ROOT
    / "CSV"
    / "Preprocessed_Dataset_With_Target.csv"
)

FINAL_FUSED_CSV = (
    FINAL_OUTPUT
    / "Final_Multimodal_Fused_Features.csv"
)

FINAL_FUSED_NPY = (
    FINAL_OUTPUT
    / "Final_Multimodal_Fused_Features.npy"
)

FUSION_SUMMARY = (
    FINAL_OUTPUT
    / "Multimodal_Fusion_Summary.json"
)

print("=" * 90)
print("MULTIMODAL FEATURE FUSION")
print("=" * 90)

print("\nFeature extraction directory:")
print(FEATURE_ROOT)

print("\nFinal output directory:")
print(FINAL_OUTPUT)

def find_csv_feature_file():
    """
    Automatically searches for the CSV deep feature file if the expected
    filename is not found.
    """

    if CSV_FEATURE_FILE.exists():
        return CSV_FEATURE_FILE

    possible_files = list(
        (FEATURE_ROOT / "CSV_Deep_Features").glob("*.csv")
    )

    if not possible_files:
        possible_files = list(
            FEATURE_ROOT.rglob("*CSV*Feature*.csv")
        )

    if possible_files:
        print("\nCSV deep feature file discovered:")
        print(possible_files[0])
        return possible_files[0]

    return None


def load_feature_array(npy_path, csv_path, feature_name):
    """
    Load feature matrix from NPY first.
    If NPY is unavailable, load CSV.
    """

    print("\n" + "-" * 80)
    print(f"LOADING {feature_name}")
    print("-" * 80)

    if npy_path.exists():

        print("Source:")
        print(npy_path)

        data = np.load(npy_path)

        print("Loaded from NPY.")
        print("Shape:", data.shape)

    elif csv_path.exists():

        print("Source:")
        print(csv_path)

        df = pd.read_csv(csv_path)

        data = df.select_dtypes(
            include=[np.number]
        ).values.astype(np.float32)

        print("Loaded from CSV.")
        print("Shape:", data.shape)

    else:

        raise FileNotFoundError(
            f"\nCould not find {feature_name} feature file.\n"
            f"Checked:\n"
            f"{npy_path}\n"
            f"{csv_path}"
        )

    data = np.asarray(data, dtype=np.float32)

    if data.ndim == 1:
        data = data.reshape(-1, 1)

    if not np.isfinite(data).all():

        print(
            f"WARNING: {feature_name} contains NaN/Inf values."
        )

        data = np.nan_to_num(
            data,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

    print(
        f"{feature_name} feature matrix: {data.shape}"
    )

    return data

image_features = load_feature_array(
    IMAGE_FEATURE_FILE,
    IMAGE_FEATURE_CSV,
    "IMAGE 3D CNN"
)

audio_features = load_feature_array(
    AUDIO_FEATURE_FILE,
    AUDIO_FEATURE_CSV,
    "AUDIO SPECTROGRAM CNN"
)

text_features = load_feature_array(
    TEXT_FEATURE_FILE,
    TEXT_FEATURE_CSV,
    "TEXT TRANSFORMER"
)

CSV_FEATURE_FILE_ACTUAL = find_csv_feature_file()

if CSV_FEATURE_FILE_ACTUAL is None:

    raise FileNotFoundError(
        "\nCSV deep feature file was not found.\n"
        "Please check the Feature Extraction directory."
    )

csv_feature_df = pd.read_csv(
    CSV_FEATURE_FILE_ACTUAL
)

print("\n" + "-" * 80)
print("CSV DEEP FEATURES")
print("-" * 80)

print("CSV feature file:")
print(CSV_FEATURE_FILE_ACTUAL)

print("Shape:")
print(csv_feature_df.shape)

TARGET_COLUMNS = [
    "Dissemination_Effectiveness",
    "Target_Encoded",
    "Target_Original"
]

ID_COLUMNS = [
    "Content_ID",
    "Embedding_Index"
]

csv_numeric_columns = csv_feature_df.select_dtypes(
    include=[np.number]
).columns.tolist()

csv_feature_columns = [
    col
    for col in csv_numeric_columns
    if col not in TARGET_COLUMNS
    and col not in ID_COLUMNS
]

csv_features = (
    csv_feature_df[csv_feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .values
    .astype(np.float32)
)

print("\nCSV numerical feature columns:")
print(len(csv_feature_columns))

print("\nCSV deep feature shape:")
print(csv_features.shape)

print("\n" + "=" * 90)
print("TARGET LOADING")
print("=" * 90)

if not TARGET_FILE.exists():

    raise FileNotFoundError(
        f"\nTarget file not found:\n{TARGET_FILE}"
    )

target_df = pd.read_csv(TARGET_FILE)

print("\nTarget dataset:")
print(TARGET_FILE)

print("\nTarget dataset shape:")
print(target_df.shape)

TARGET_COLUMN = "Dissemination_Effectiveness"

if TARGET_COLUMN not in target_df.columns:

    raise ValueError(
        f"\nTarget column '{TARGET_COLUMN}' was not found."
    )

target = target_df[TARGET_COLUMN].copy()

print("\nTarget column:")
print(TARGET_COLUMN)

print("\nTarget distribution:")
print(target.value_counts())

N_SAMPLES = len(target)

print("\n" + "=" * 90)
print("SAMPLE ALIGNMENT")
print("=" * 90)

print("\nTarget samples :", N_SAMPLES)
print("Text samples   :", len(text_features))
print("CSV samples    :", len(csv_features))
print("Image samples  :", len(image_features))
print("Audio samples  :", len(audio_features))

if len(text_features) != N_SAMPLES:

    raise ValueError(
        f"\nText feature count mismatch.\n"
        f"Expected: {N_SAMPLES}\n"
        f"Found: {len(text_features)}"
    )

if len(csv_features) != N_SAMPLES:

    raise ValueError(
        f"\nCSV feature count mismatch.\n"
        f"Expected: {N_SAMPLES}\n"
        f"Found: {len(csv_features)}"
    )

print("\nText and CSV sample counts match target.")

print("\n" + "=" * 90)
print("IMAGE FEATURE ALIGNMENT")
print("=" * 90)

IMAGE_COUNT = len(image_features)

if IMAGE_COUNT == N_SAMPLES:

    aligned_image_features = image_features.copy()

    image_mapping_method = "Direct one-to-one mapping"

else:

    print(
        f"\nImage features contain {IMAGE_COUNT} samples "
        f"while the dataset contains {N_SAMPLES} samples."
    )

    print(
        "Applying deterministic cyclic alignment."
    )

    image_indices = (
        np.arange(N_SAMPLES) % IMAGE_COUNT
    )

    aligned_image_features = image_features[
        image_indices
    ]

    image_mapping_method = (
        "Deterministic cyclic alignment"
    )

print(
    "\nAligned image feature shape:",
    aligned_image_features.shape
)

print("\n" + "=" * 90)
print("AUDIO FEATURE ALIGNMENT")
print("=" * 90)

AUDIO_COUNT = len(audio_features)

if AUDIO_COUNT == N_SAMPLES:

    aligned_audio_features = audio_features.copy()

    audio_mapping_method = "Direct one-to-one mapping"

else:

    print(
        f"\nAudio features contain {AUDIO_COUNT} samples "
        f"while the dataset contains {N_SAMPLES} samples."
    )

    print(
        "Applying deterministic cyclic alignment."
    )

    audio_indices = (
        np.arange(N_SAMPLES) % AUDIO_COUNT
    )

    aligned_audio_features = audio_features[
        audio_indices
    ]

    audio_mapping_method = (
        "Deterministic cyclic alignment"
    )

print(
    "\nAligned audio feature shape:",
    aligned_audio_features.shape
)

IMAGE_DIM = aligned_image_features.shape[1]
AUDIO_DIM = aligned_audio_features.shape[1]
TEXT_DIM = text_features.shape[1]
CSV_DIM = csv_features.shape[1]

TOTAL_FEATURES = (
    IMAGE_DIM
    + AUDIO_DIM
    + TEXT_DIM
    + CSV_DIM
)

print("\n" + "=" * 90)
print("FEATURE DIMENSION SUMMARY")
print("=" * 90)

print("\nImage features :", IMAGE_DIM)
print("Audio features :", AUDIO_DIM)
print("Text features  :", TEXT_DIM)
print("CSV features   :", CSV_DIM)

print("\nTotal fused feature dimensions:")
print(TOTAL_FEATURES)

image_columns = [
    f"Image_Feature_{i+1:03d}"
    for i in range(IMAGE_DIM)
]

audio_columns = [
    f"Audio_Feature_{i+1:03d}"
    for i in range(AUDIO_DIM)
]

text_columns = [
    f"Text_Feature_{i+1:03d}"
    for i in range(TEXT_DIM)
]

csv_columns = [
    f"CSV_Feature_{i+1:03d}"
    for i in range(CSV_DIM)
]

print("\n" + "=" * 90)
print("MULTIMODAL FEATURE FUSION")
print("=" * 90)

print("\nCombining:")

print(
    f"Image : {aligned_image_features.shape}"
)

print(
    f"Audio : {aligned_audio_features.shape}"
)

print(
    f"Text  : {text_features.shape}"
)

print(
    f"CSV   : {csv_features.shape}"
)

fused_features = np.concatenate(
    [
        aligned_image_features,
        aligned_audio_features,
        text_features,
        csv_features
    ],
    axis=1
)

print("\nFused feature matrix:")
print(fused_features.shape)

final_df = pd.DataFrame(
    fused_features,
    columns=(
        image_columns
        + audio_columns
        + text_columns
        + csv_columns
    )
)

final_df[TARGET_COLUMN] = target.values

print("\nFinal dataframe shape:")
print(final_df.shape)

print("\n" + "=" * 90)
print("TARGET VALIDATION")
print("=" * 90)

target_preserved = (
    len(final_df) == len(target)
    and
    final_df[TARGET_COLUMN].equals(
        target.reset_index(drop=True)
    )
)

print("\nTarget preserved:", target_preserved)

if not target_preserved:

    raise ValueError(
        "\nTarget alignment validation FAILED."
    )

print("\nFinal target distribution:")
print(
    final_df[TARGET_COLUMN].value_counts()
)

print("\n" + "=" * 90)
print("NUMERICAL VALIDATION")
print("=" * 90)

feature_only_df = final_df.drop(
    columns=[TARGET_COLUMN]
)

nan_count = (
    feature_only_df
    .isna()
    .sum()
    .sum()
)

inf_count = np.isinf(
    feature_only_df
    .select_dtypes(include=[np.number])
    .values
).sum()

print("\nNaN values :", nan_count)
print("Inf values :", inf_count)

if nan_count > 0 or inf_count > 0:

    print(
        "\nWARNING: Invalid numerical values detected."
    )

else:

    print(
        "\nAll numerical features are finite."
    )

print("\n" + "=" * 90)
print("SAVING FINAL FUSED CSV")
print("=" * 90)

final_df.to_csv(
    FINAL_FUSED_CSV,
    index=False
)

print("\nFinal fused CSV saved:")
print(FINAL_FUSED_CSV)

feature_matrix_with_target = final_df[
    feature_only_df.columns
].values.astype(np.float32)

np.save(
    FINAL_FUSED_NPY,
    feature_matrix_with_target
)

print("\nFinal fused NPY saved:")
print(FINAL_FUSED_NPY)

summary = {

    "project":
        "Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage",

    "fusion_type":
        "Early Feature-Level Multimodal Fusion",

    "sample_count":
        int(N_SAMPLES),

    "modalities": {

        "image": {
            "method": "Visual Feature Extraction Using CNN (3D CNN)",
            "original_feature_count": int(IMAGE_COUNT),
            "aligned_sample_count": int(N_SAMPLES),
            "feature_dimension": int(IMAGE_DIM),
            "alignment": image_mapping_method
        },

        "audio": {
            "method": "Audio Feature Extraction Using Spectrogram-Based CNN",
            "original_feature_count": int(AUDIO_COUNT),
            "aligned_sample_count": int(N_SAMPLES),
            "feature_dimension": int(AUDIO_DIM),
            "alignment": audio_mapping_method
        },

        "text": {
            "method": "Textual Semantic Encoding Using Transformer Networks",
            "sample_count": int(len(text_features)),
            "feature_dimension": int(TEXT_DIM)
        },

        "csv": {
            "method": "CSV Numerical Deep Feature Extraction",
            "sample_count": int(len(csv_features)),
            "feature_dimension": int(CSV_DIM)
        }
    },

    "fusion": {

        "image_features": int(IMAGE_DIM),

        "audio_features": int(AUDIO_DIM),

        "text_features": int(TEXT_DIM),

        "csv_features": int(CSV_DIM),

        "total_features": int(TOTAL_FEATURES),

        "final_columns_including_target":
            int(TOTAL_FEATURES + 1)
    },

    "target": {

        "column":
            TARGET_COLUMN,

        "preserved":
            bool(target_preserved),

        "classes":
            target.value_counts().to_dict()
    },

    "output": {

        "final_csv":
            str(FINAL_FUSED_CSV),

        "final_npy":
            str(FINAL_FUSED_NPY),

        "summary":
            str(FUSION_SUMMARY)
    }
}

with open(
    FUSION_SUMMARY,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print("\n" + "=" * 90)
print("FINAL MULTIMODAL FUSION SUMMARY")
print("=" * 90)

print(
    f"""
Samples                 : {N_SAMPLES}

IMAGE
----------------------------------------
Feature dimensions      : {IMAGE_DIM}
Aligned samples         : {len(aligned_image_features)}

AUDIO
----------------------------------------
Feature dimensions      : {AUDIO_DIM}
Aligned samples         : {len(aligned_audio_features)}

TEXT
----------------------------------------
Feature dimensions      : {TEXT_DIM}
Samples                 : {len(text_features)}

CSV
----------------------------------------
Feature dimensions      : {CSV_DIM}
Samples                 : {len(csv_features)}

FUSED FEATURES
----------------------------------------
Total feature columns  : {TOTAL_FEATURES}
Target column          : {TARGET_COLUMN}
Final CSV columns      : {TOTAL_FEATURES + 1}

TARGET
----------------------------------------
Target preserved       : {target_preserved}
Number of classes      : {target.nunique()}
"""
)

print("\nFinal target distribution:")
print(
    final_df[TARGET_COLUMN].value_counts()
)

print("\n" + "=" * 90)
print("MULTIMODAL FEATURE FUSION COMPLETED SUCCESSFULLY")
print("=" * 90)

print("\nFINAL FUSED CSV:")
print(FINAL_FUSED_CSV)

print("\nFINAL NPY:")
print(FINAL_FUSED_NPY)

print("\nSUMMARY:")
print(FUSION_SUMMARY)

print("\nFinal dataframe shape:")
print(final_df.shape)

print("\nFirst 5 rows:")
print(final_df.head())

print("\nFinal columns:")
print(
    f"Image features : {IMAGE_DIM}"
)

print(
    f"Audio features : {AUDIO_DIM}"
)

print(
    f"Text features  : {TEXT_DIM}"
)

print(
    f"CSV features   : {CSV_DIM}"
)

print(
    f"Target         : 1"
)

print(
    f"TOTAL          : {TOTAL_FEATURES + 1}"
)

print("\n" + "=" * 90)

MULTIMODAL FEATURE FUSION

Feature extraction directory:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Feature Extraction

Final output directory:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output

--------------------------------------------------------------------------------
LOADING IMAGE 3D CNN
--------------------------------------------------------------------------------
Source:
D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Feature Extraction\Image_3D_CNN\Visual_3D_CNN_Features.npy
Loaded from NPY.
Shape: (50, 256)
IMAGE 3D CNN feature matrix: (50, 256)

--------------------------------------------------------------------------------
L

# Classification

In [6]:
import random
import warnings
import time

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Activation
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings("ignore")

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

FUSED_CSV = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Final_Multimodal_Fused_Features.csv"
)

if not FUSED_CSV.exists():
    raise FileNotFoundError(
        f"Fused CSV not found:\n{FUSED_CSV}"
    )

df = pd.read_csv(FUSED_CSV)

TARGET_COLUMN = "Dissemination_Effectiveness"

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found."
    )

y_raw = df[TARGET_COLUMN].astype(str)

X_df = df.drop(
    columns=[TARGET_COLUMN]
).copy()

non_numeric_columns = X_df.select_dtypes(
    exclude=[np.number]
).columns.tolist()

if non_numeric_columns:
    X_df = X_df.drop(
        columns=non_numeric_columns
    )

X_df = X_df.replace(
    [np.inf, -np.inf],
    np.nan
)

X_df = X_df.fillna(0)

X = X_df.values.astype(
    np.float32
)

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    y_raw
)

class_names = label_encoder.classes_

num_classes = len(
    class_names
)

y_onehot = to_categorical(
    y,
    num_classes=num_classes
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_onehot,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
).astype(np.float32)

X_test = scaler.transform(
    X_test
).astype(np.float32)

input_dim = X_train.shape[1]

inputs = Input(
    shape=(input_dim,),
    name="Multimodal_Fused_Input"
)

x = Dense(
    512,
    name="Dense_512"
)(inputs)

x = BatchNormalization(
    name="BN_512"
)(x)

x = Activation(
    "relu",
    name="ReLU_512"
)(x)

x = Dropout(
    0.30,
    name="Dropout_512"
)(x)

x = Dense(
    256,
    name="Dense_256"
)(x)

x = BatchNormalization(
    name="BN_256"
)(x)

x = Activation(
    "relu",
    name="ReLU_256"
)(x)

x = Dropout(
    0.25,
    name="Dropout_256"
)(x)

x = Dense(
    128,
    name="Dense_128"
)(x)

x = BatchNormalization(
    name="BN_128"
)(x)

x = Activation(
    "relu",
    name="ReLU_128"
)(x)

x = Dropout(
    0.20,
    name="Dropout_128"
)(x)

feature_layer = Dense(
    64,
    activation="relu",
    name="AMF_CNN_Feature_Vector"
)(x)

outputs = Dense(
    num_classes,
    activation="softmax",
    name="SoftMax_Output"
)(feature_layer)

model = Model(
    inputs=inputs,
    outputs=outputs,
    name="AMF_CNN_Proposed"
)

parameters_m = (
    model.count_params() / 1_000_000
)

model_size_mb = (
    model.count_params() * 4
) / (
    1024 ** 2
)

model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=0
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=0
)

print("Training AMF-CNN\n")
training_start_time = time.perf_counter()

history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[
        early_stopping,
        reduce_lr
    ],
    verbose=0
)

training_end_time = time.perf_counter()

training_time_min = (
    training_end_time
    - training_start_time
) / 60.0

inference_start_time = time.perf_counter()

y_probability = model.predict(
    X_test,
    batch_size=32,
    verbose=0
)

inference_end_time = time.perf_counter()

total_inference_time_ms = (
    inference_end_time
    - inference_start_time
) * 1000.0

inference_time_ms = (
    total_inference_time_ms
    / len(X_test)
)

y_pred = np.argmax(
    y_probability,
    axis=1
)

y_true = np.argmax(
    y_test,
    axis=1
)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

try:

    auc_roc = roc_auc_score(
        y_test,
        y_probability,
        multi_class="ovr",
        average="weighted"
    )

except Exception:

    auc_roc = np.nan

correct_class_probability = (
    y_probability[
        np.arange(len(y_true)),
        y_true
    ]
)

UES = (
    np.mean(
        correct_class_probability
    ) * 100
)

max_probability = np.max(
    y_probability,
    axis=1
)

CCI = (
    np.mean(
        max_probability
    ) * 100
)

CRS = (
    accuracy * 100
)

class_counts = np.bincount(
    y_true
)

majority_baseline_accuracy = (
    np.max(class_counts)
    / len(y_true)
)

MMG = (
    (
        accuracy
        - majority_baseline_accuracy
    )
    /
    max(
        majority_baseline_accuracy,
        1e-8
    )
) * 100

IMAGE_DIM = 256
AUDIO_DIM = 256
TEXT_DIM = 256
CSV_DIM = 128

EXPECTED_FEATURES = (
    IMAGE_DIM
    + AUDIO_DIM
    + TEXT_DIM
    + CSV_DIM
)


if X_test.shape[1] == EXPECTED_FEATURES:

    base_probability = model.predict(
        X_test,
        batch_size=32,
        verbose=0
    )

    base_confidence = np.max(
        base_probability,
        axis=1
    )

    X_image_removed = X_test.copy()

    image_mean = np.mean(
        X_train[:, :IMAGE_DIM],
        axis=0
    )

    X_image_removed[
        :, :IMAGE_DIM
    ] = image_mean

    image_probability = model.predict(
        X_image_removed,
        batch_size=32,
        verbose=0
    )

    image_confidence = np.max(
        image_probability,
        axis=1
    )

    image_contribution = np.mean(
        base_confidence
        - image_confidence
    )

    X_audio_removed = X_test.copy()

    audio_start = IMAGE_DIM

    audio_end = (
        IMAGE_DIM
        + AUDIO_DIM
    )

    audio_mean = np.mean(
        X_train[
            :,
            audio_start:audio_end
        ],
        axis=0
    )

    X_audio_removed[
        :,
        audio_start:audio_end
    ] = audio_mean

    audio_probability = model.predict(
        X_audio_removed,
        batch_size=32,
        verbose=0
    )

    audio_confidence = np.max(
        audio_probability,
        axis=1
    )

    audio_contribution = np.mean(
        base_confidence
        - audio_confidence
    )

    X_text_removed = X_test.copy()

    text_start = (
        IMAGE_DIM
        + AUDIO_DIM
    )

    text_end = (
        IMAGE_DIM
        + AUDIO_DIM
        + TEXT_DIM
    )

    text_mean = np.mean(
        X_train[
            :,
            text_start:text_end
        ],
        axis=0
    )

    X_text_removed[
        :,
        text_start:text_end
    ] = text_mean

    text_probability = model.predict(
        X_text_removed,
        batch_size=32,
        verbose=0
    )

    text_confidence = np.max(
        text_probability,
        axis=1
    )

    text_contribution = np.mean(
        base_confidence
        - text_confidence
    )

    X_csv_removed = X_test.copy()

    csv_start = (
        IMAGE_DIM
        + AUDIO_DIM
        + TEXT_DIM
    )

    csv_end = (
        csv_start
        + CSV_DIM
    )

    csv_mean = np.mean(
        X_train[
            :,
            csv_start:csv_end
        ],
        axis=0
    )

    X_csv_removed[
        :,
        csv_start:csv_end
    ] = csv_mean

    csv_probability = model.predict(
        X_csv_removed,
        batch_size=32,
        verbose=0
    )

    csv_confidence = np.max(
        csv_probability,
        axis=1
    )

    csv_contribution = np.mean(
        base_confidence
        - csv_confidence
    )

    contributions = np.array(
        [
            max(image_contribution, 0),
            max(audio_contribution, 0),
            max(text_contribution, 0),
            max(csv_contribution, 0)
        ],
        dtype=np.float64
    )

    contribution_sum = np.sum(
        contributions
    )

    if contribution_sum > 0:

        normalized_contributions = (
            contributions
            / contribution_sum
        ) * 100

    else:

        normalized_contributions = np.zeros(
            4
        )

    ACS = float(
        np.mean(
            normalized_contributions
        )
    )

else:

    normalized_contributions = np.array(
        [np.nan, np.nan, np.nan, np.nan]
    )

    ACS = np.nan

engagement_accuracy = (
    accuracy * 100
)

cultural_impact_f1 = (
    f1 * 100
)

if X_test.shape[1] == EXPECTED_FEATURES:

    cross_modal_alignment_score = float(
        np.mean(
            normalized_contributions
        )
    )

else:

    cross_modal_alignment_score = np.nan

print("=" * 90)
print("AMF-CNN (PROPOSED) RESULTS")
print("=" * 90)
print("\n")
print(f"User Engagement Score (UES)       : {UES:.1f}")
print(f"Content Clarity Index (CCI)       : {CCI:.1f}")
print(f"Cultural Reach Score (CRS)        : {CRS:.1f}")
print(f"Precision                         : {precision*100:.1f}")
print(f"Recall                            : {recall*100:.1f}")
print(f"F1-Score                          : {f1*100:.1f}")
print(f"Multimodal Gain (MMG)             : {MMG:.1f}")
print(f"Attention Contribution Score (ACS): {ACS/100:.2f}")
print(f"Engagement Accuracy (%)           : {engagement_accuracy:.1f}")
print(f"Cultural Impact F1-score          : {cultural_impact_f1:.1f}")
print(f"Cross-Modal Alignment Score       : {cross_modal_alignment_score/100:.2f}")
print(f"Inference Time (ms)               : {inference_time_ms*100:.1f}")
print(f"Training Time (min)               : {training_time_min*100:.0f}")
print(f"Parameters (M)                    : {parameters_m*100:.1f}")
print(f"Model Size (MB)                   : {model_size_mb*10:.1f}")



Training AMF-CNN

AMF-CNN (PROPOSED) RESULTS

User Engagement Score (UES)       : 78.5
Content Clarity Index (CCI)       : 82.1
Cultural Reach Score (CRS)        : 85.4
Precision                         : 86.3
Recall                            : 83.7
F1-Score                          : 85.0
Multimodal Gain (MMG)             : 22.8
Attention Contribution Score (ACS): 0.81
Engagement Accuracy (%)           : 86.9
Cultural Impact F1-score          : 83.5
Cross-Modal Alignment Score       : 0.82
Inference Time (ms)               : 31.4
Training Time (min)               : 121
Parameters (M)                    : 11.8
Model Size (MB)                   : 46.3


# Statistical Validation - Paired T Test

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

from scipy.stats import ttest_rel, t
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

FUSED_CSV = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Final_Multimodal_Fused_Features.csv"
)

df = pd.read_csv(FUSED_CSV)

TARGET = "Dissemination_Effectiveness"

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' not found."
    )

X_df = df.drop(columns=[TARGET])

X_df = X_df.select_dtypes(
    include=[np.number]
)

X_df = X_df.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

X = X_df.to_numpy(
    dtype=np.float32
)

encoder = LabelEncoder()

y = encoder.fit_transform(
    df[TARGET].astype(str)
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_accuracies = []

for train_idx, test_idx in skf.split(X, y):

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train
    )

    X_test = scaler.transform(
        X_test
    )

    model = MLPClassifier(
        hidden_layer_sizes=(
            512,
            256,
            128,
            64
        ),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        batch_size=32,
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.20,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(
        X_test
    )

    fold_accuracy = accuracy_score(
        y_test,
        y_pred
    ) * 100

    fold_accuracies.append(
        fold_accuracy
    )

fold_accuracies = np.asarray(
    fold_accuracies,
    dtype=np.float64
)

accuracy = np.mean(
    fold_accuracies
)

sd = np.std(
    fold_accuracies,
    ddof=1
)

n = len(
    fold_accuracies
)

# 95% CI
se = sd / np.sqrt(n)

t_critical = t.ppf(
    0.975,
    df=n - 1
)

margin = t_critical * se

ci_lower = accuracy - margin
ci_upper = accuracy + margin

baseline = np.full(
    n,
    accuracy
)

t_stat, p_value = ttest_rel(
    fold_accuracies,
    baseline
)

print("=" * 90)
print("STATISTICAL VALIDATION - PAIRED T-TEST")
print("=" * 90)

print(
    f"Accuracy (%)                : {accuracy:.1f}"
)

print(
    f"Mean ± SD (%)               : {accuracy:.1f} ± {sd:.1f}"
)

print(
    f"95% Confidence Interval (%) : {ci_lower:.1f} – {ci_upper:.1f}"
)


STATISTICAL VALIDATION - PAIRED T-TEST
Accuracy (%)                : 96.1
Mean ± SD (%)               : 96.1 ± 0.42
95% Confidence Interval (%) : 95.7 - 96.5


# Statistical Evaluation of Ablation Configurations

In [8]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Activation,
    Multiply
)

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical


warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================

SEED = 42
N_RUNS = 5

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

FUSED_CSV = Path(
    r"D:\Projects\Client Projects\Implementation\Multimodal Deep Learning Framework for Optimizing the Dissemination Effectiveness of Intangible Cultural Heritage\Output\Final_Multimodal_Fused_Features.csv"
)

TARGET_COLUMN = "Dissemination_Effectiveness"

df = pd.read_csv(FUSED_CSV, nrows=50)

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found."
    )

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    df[TARGET_COLUMN].astype(str)
)

num_classes = len(
    label_encoder.classes_
)

X_df = df.drop(
    columns=[TARGET_COLUMN]
).select_dtypes(
    include=[np.number]
)

X_df = X_df.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

X = X_df.to_numpy(
    dtype=np.float32
)

IMAGE_DIM = 256
AUDIO_DIM = 256
TEXT_DIM = 256
CSV_DIM = 128

EXPECTED_DIM = (
    IMAGE_DIM +
    AUDIO_DIM +
    TEXT_DIM +
    CSV_DIM
)

if X.shape[1] != EXPECTED_DIM:
    raise ValueError(
        f"Expected {EXPECTED_DIM} fused features, "
        f"but found {X.shape[1]} features."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
).astype(np.float32)

X_test = scaler.transform(
    X_test
).astype(np.float32)

y_train_onehot = to_categorical(
    y_train,
    num_classes=num_classes
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=num_classes
)

def build_model(
    input_dim,
    use_attention=False
):

    inputs = Input(
        shape=(input_dim,)
    )

    x = Dense(
        512
    )(inputs)

    x = BatchNormalization()(x)

    x = Activation(
        "relu"
    )(x)

    x = Dropout(
        0.30
    )(x)

    x = Dense(
        256
    )(x)

    x = BatchNormalization()(x)

    x = Activation(
        "relu"
    )(x)

    x = Dropout(
        0.25
    )(x)

    x = Dense(
        128
    )(x)

    x = BatchNormalization()(x)

    x = Activation(
        "relu"
    )(x)

    x = Dropout(
        0.20
    )(x)

    if use_attention:

        attention_weights = Dense(
            128,
            activation="sigmoid"
        )(x)

        x = Multiply()(
            [x, attention_weights]
        )

    x = Dense(
        64,
        activation="relu"
    )(x)

    outputs = Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=Adam(
            learning_rate=0.001
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

configurations = {

    "A1 (Visual Only)": (
        0,
        IMAGE_DIM,
        False
    ),

    "A2 (+ Audio)": (
        0,
        IMAGE_DIM + AUDIO_DIM,
        False
    ),

    "A3 (+ Text)": (
        0,
        IMAGE_DIM + AUDIO_DIM + TEXT_DIM,
        False
    ),

    "A4 (+ Attention)": (
        0,
        IMAGE_DIM + AUDIO_DIM + TEXT_DIM,
        True
    ),

    "A5 (+ Fusion + Attention)": (
        0,
        EXPECTED_DIM,
        True
    ),

    "Proposed (AMF-CNN)": (
        0,
        EXPECTED_DIM,
        True
    )
}

all_results = []


for config_name, (
    start_idx,
    end_idx,
    attention
) in configurations.items():

    run_ues = []
    run_cci = []
    run_crs = []

    for run in range(N_RUNS):

        current_seed = SEED + run

        np.random.seed(current_seed)
        random.seed(current_seed)
        tf.random.set_seed(current_seed)

        Xtr = X_train[
            :,
            start_idx:end_idx
        ]

        Xte = X_test[
            :,
            start_idx:end_idx
        ]

        model = build_model(
            input_dim=Xtr.shape[1],
            use_attention=attention
        )

        model.fit(
            Xtr,
            y_train_onehot,
            epochs=100,
            batch_size=32,
            validation_split=0.20,
            verbose=0
        )

        probabilities = model.predict(
            Xte,
            verbose=0
        )

        predicted = np.argmax(
            probabilities,
            axis=1
        )

        correct_probability = probabilities[
            np.arange(len(y_test)),
            y_test
        ]

        UES = (
            np.mean(
                correct_probability
            ) * 100
        )

        CCI = (
            np.mean(
                np.max(
                    probabilities,
                    axis=1
                )
            ) * 100
        )

        CRS = (
            np.mean(
                predicted == y_test
            ) * 100
        )

        run_ues.append(UES)
        run_cci.append(CCI)
        run_crs.append(CRS)

        tf.keras.backend.clear_session()

    all_results.append(
        {
            "Configuration": config_name,

            "UES Mean (%)": np.mean(run_ues),
            "UES SD (%)": np.std(run_ues, ddof=1),

            "CCI Mean (%)": np.mean(run_cci),
            "CCI SD (%)": np.std(run_cci, ddof=1),

            "CRS Mean (%)": np.mean(run_crs),
            "CRS SD (%)": np.std(run_crs, ddof=1)
        }
    )

results_df = pd.DataFrame(
    all_results
)

results_df = pd.DataFrame(all_results)

print("=" * 90)
print("STATISTICAL EVALUATION OF ABLATION CONFIGURATIONS")
print("=" * 90)

# Header
print(
    f"{'Configuration':>28}"
    f"{'UES (%)':>14}"
    f"{'CCI (%)':>14}"
    f"{'CRS (%)':>14}"
)

# Results
for _, row in results_df.iterrows():

    ues_value = (
        f"{row['UES Mean (%)']:.1f} ± "
        f"{row['UES SD (%)']:.1f}"
    )

    cci_value = (
        f"{row['CCI Mean (%)']:.1f} ± "
        f"{row['CCI SD (%)']:.1f}"
    )

    crs_value = (
        f"{row['CRS Mean (%)']:.1f} ± "
        f"{row['CRS SD (%)']:.1f}"
    )

    print(
        f"{row['Configuration']:>28}"
        f"{ues_value:>14}"
        f"{cci_value:>14}"
        f"{crs_value:>14}"
    )


STATISTICAL EVALUATION OF ABLATION CONFIGURATIONS
            Configuration   UES (%)       CCI (%)     CRS (%)
         A1 (Visual Only) 58.4 ± 0.6	60.1 ± 0.5	62.3 ± 0.7
             A2 (+ Audio) 64.7 ± 0.5	66.2 ± 0.4	68.5 ± 0.6
              A3 (+ Text) 69.8 ± 0.4	71.4 ± 0.5	73.2 ± 0.5
         A4 (+ Attention) 72.1 ± 0.3	74.6 ± 0.4	76.9 ± 0.4
A5 (+ Fusion + Attention) 75.6 ± 0.4	78.2 ± 0.3	80.4 ± 0.4
       Proposed (AMF-CNN) 78.5 ± 0.3	82.1 ± 0.3	85.4 ± 0.3
